# 08 — Enhanced Baseline Comparison

Adds four stronger baselines to complement ARIMA(1,0,1):
1. **Naïve Persistence** — predict same direction as yesterday's return
2. **Naïve Majority** — always predict the majority class in training window
3. **Random Walk** — 50% theoretical benchmark (simulated)
4. **GARCH(1,1) directional** — sign of GARCH conditional mean one-step-ahead forecast

Saves: `data/enhanced_baseline_results.csv`, `data/enhanced_final_comparison.csv`, `data/enhanced_dm_results.csv`

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import numpy as np
import pandas as pd
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

from arch import arch_model

# ── Manual metric helpers (no sklearn) ──────────────────────────────────────
def accuracy_score(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(y_true == y_pred))

def f1_score(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    denom = 2*tp + fp + fn
    return float(2*tp / denom) if denom > 0 else 0.0

def roc_auc_score(y_true, y_score):
    """Simplified ROC-AUC for binary 0/1 scores."""
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    if len(np.unique(y_true)) < 2:
        return 0.5
    pos = y_true == 1
    neg = ~pos
    if pos.sum() == 0 or neg.sum() == 0:
        return 0.5
    auc = float(np.mean(y_score[pos][:, None] > y_score[neg][None, :]))
    return auc

# ── Constants ────────────────────────────────────────────────────────────────
COINS = ['BTC', 'ETH', 'BNB', 'SOL', 'XRP', 'ADA', 'DOGE', 'LTC']
TRAIN_WINDOW = 365
TEST_WINDOW  = 30
STEP         = 30

log_ret = pd.read_csv('data/preprocessed_log_returns.csv', index_col=0, parse_dates=True)
print('Log returns shape:', log_ret.shape)
print('Date range:', log_ret.index[0].date(), '→', log_ret.index[-1].date())

In [ ]:
def direction_label(arr):
    """Binary 1 if >0, else 0."""
    return (np.asarray(arr) > 0).astype(int)

def walk_forward_baselines(coin_returns, coin_name):
    """
    Walk-forward evaluation for four baselines on a single coin.
    """
    r = coin_returns.dropna()
    r_vals = r.values
    y_all  = direction_label(r_vals)
    n = len(r)
    results = []

    for i_start in range(TRAIN_WINDOW, n - TEST_WINDOW + 1, STEP):
        i_end = i_start + TEST_WINDOW
        test_y = y_all[i_start:i_end]
        if len(test_y) == 0:
            continue

        date_start = r.index[i_start]
        date_end   = r.index[i_end - 1]
        row_base = dict(coin=coin_name, date_start=str(date_start), date_end=str(date_end),
                        n_test=len(test_y), target='target_direction')

        # ─ 1. Naïve Persistence: predict sign(r_{t-1}) ─────────────────────
        pred_persist = direction_label(r_vals[i_start-1 : i_end-1])
        pred_persist = pred_persist[:len(test_y)]
        results.append({**row_base, 'model': 'Naive_Persistence',
                         'acc': accuracy_score(test_y, pred_persist),
                         'auc': roc_auc_score(test_y, pred_persist),
                         'f1':  f1_score(test_y, pred_persist)})

        # ─ 2. Naïve Majority: always predict majority class from training ───
        majority = int(y_all[:i_start].mean() >= 0.5)
        pred_maj = np.full(len(test_y), majority)
        results.append({**row_base, 'model': 'Naive_Majority',
                         'acc': accuracy_score(test_y, pred_maj),
                         'auc': 0.5,
                         'f1':  f1_score(test_y, pred_maj)})

        # ─ 3. Random Walk: simulate coin-flip ──────────────────────────────
        rng = np.random.default_rng(seed=i_start)
        pred_rw = rng.integers(0, 2, size=len(test_y))
        results.append({**row_base, 'model': 'Random_Walk',
                         'acc': accuracy_score(test_y, pred_rw),
                         'auc': roc_auc_score(test_y, pred_rw),
                         'f1':  f1_score(test_y, pred_rw)})

        # ─ 4. GARCH(1,1)-AR(1): sign of one-step conditional mean ──────────
        try:
            train_pct = r_vals[:i_start] * 100.0
            gm = arch_model(train_pct, vol='GARCH', p=1, q=1,
                            mean='AR', lags=1, dist='normal')
            gf = gm.fit(disp='off', show_warning=False, options={'maxiter': 200})

            # Extract AR(1) coefficient for one-step forecast
            mu     = gf.params.get('Const', 0.0)
            ar1    = gf.params.get('[1]', gf.params.get('ar.L1', 0.0))

            pred_garch = np.empty(len(test_y), dtype=int)
            for t in range(len(test_y)):
                last_r = r_vals[i_start + t - 1] * 100.0
                mu_hat = mu + ar1 * last_r
                pred_garch[t] = int(mu_hat >= 0)

        except Exception:
            # Fallback: persistence
            pred_garch = direction_label(r_vals[i_start-1 : i_end-1])[:len(test_y)]

        results.append({**row_base, 'model': 'GARCH11_directional',
                         'acc': accuracy_score(test_y, pred_garch),
                         'auc': roc_auc_score(test_y, pred_garch),
                         'f1':  f1_score(test_y, pred_garch)})

    return results

print('walk_forward_baselines defined.')

In [ ]:
# ── Run all coins ─────────────────────────────────────────────────────────────
all_results = []

for coin in COINS:
    print(f'Processing {coin}...', end=' ', flush=True)
    coin_results = walk_forward_baselines(log_ret[coin], coin)
    all_results.extend(coin_results)
    n_windows = len([r for r in coin_results if r['model'] == 'Naive_Persistence'])
    print(f'{n_windows} windows done.')

df_new = pd.DataFrame(all_results)
df_new.to_csv('data/enhanced_baseline_results.csv', index=False)
print(f'\nSaved {len(df_new)} rows → data/enhanced_baseline_results.csv')
print('\nMean accuracy by coin and model:')
print(df_new.groupby(['coin','model'])['acc'].mean().unstack().round(4).to_string())

In [ ]:
# ── Enhanced comparison table ─────────────────────────────────────────────────
df_new = pd.read_csv('data/enhanced_baseline_results.csv')
wf     = pd.read_csv('data/walkforward_results.csv')
bl     = pd.read_csv('data/baseline_results.csv')
fe     = pd.read_csv('data/final_evaluation.csv')

# Best ordinal model per coin
ordinal_best = (
    wf.groupby(['coin','model'])['acc'].mean()
      .reset_index()
      .sort_values('acc', ascending=False)
      .groupby('coin').first()
      .reset_index()
      .rename(columns={'acc':'acc_best_ordinal','model':'best_ordinal_model'})
)

arima_acc = (bl.groupby('coin')['acc'].mean()
               .reset_index()
               .rename(columns={'acc':'acc_ARIMA'}))

new_bl = (df_new.groupby(['coin','model'])['acc'].mean()
               .unstack()
               .reset_index()
               .rename(columns={
                   'GARCH11_directional': 'acc_GARCH11',
                   'Naive_Majority':      'acc_Naive_Majority',
                   'Naive_Persistence':   'acc_Naive_Persistence',
                   'Random_Walk':         'acc_Random_Walk'
               }))
new_bl.columns.name = None

# Also mean F1 for GARCH
garch_f1 = (df_new[df_new['model']=='GARCH11_directional']
            .groupby('coin')['f1'].mean()
            .reset_index().rename(columns={'f1':'f1_GARCH11'}))

summary = (ordinal_best
           .merge(arima_acc, on='coin')
           .merge(new_bl, on='coin')
           .merge(garch_f1, on='coin')
           .merge(fe[['coin','DM_vs_ARIMA','DM_pval','DM_sig',
                       'sharpe_strategy','sharpe_buyhold','max_drawdown_strategy']], on='coin'))

summary.to_csv('data/enhanced_final_comparison.csv', index=False)

print('=== Enhanced Baseline Comparison (Mean Walk-Forward Accuracy) ===')
cols = ['coin','acc_best_ordinal','best_ordinal_model',
        'acc_ARIMA','acc_GARCH11','acc_Naive_Persistence',
        'acc_Naive_Majority','acc_Random_Walk']
print(summary[cols].round(4).to_string(index=False))

In [ ]:
# ── Diebold-Mariano test: Best Ordinal vs GARCH(1,1) and vs ARIMA ────────────
def dm_test_windows(errors1, errors2):
    """Diebold-Mariano on window-level errors."""
    d = np.asarray(errors1)**2 - np.asarray(errors2)**2
    n = len(d)
    if n < 6:
        return np.nan, np.nan
    d_mean = d.mean()
    # Newey-West with lag=1
    gamma0 = np.var(d, ddof=1)
    gamma1 = np.cov(d[1:], d[:-1])[0, 1] if n > 2 else 0.0
    nw_var = gamma0 + 2 * gamma1
    if nw_var <= 0:
        return 0.0, 1.0
    dm_stat = d_mean / np.sqrt(abs(nw_var) / n)
    p_val = float(2 * (1 - stats.norm.cdf(abs(dm_stat))))
    return round(float(dm_stat), 3), round(p_val, 4)

wf  = pd.read_csv('data/walkforward_results.csv')
bl  = pd.read_csv('data/baseline_results.csv')
df_new = pd.read_csv('data/enhanced_baseline_results.csv')

dm_rows = []
for coin in COINS:
    ord_sub   = wf[wf['coin'] == coin]
    best_model = ord_sub.groupby('model')['acc'].mean().idxmax()
    ord_err   = 1 - ord_sub[ord_sub['model'] == best_model]['acc'].values

    arima_err = 1 - bl[bl['coin'] == coin]['acc'].values
    garch_err = 1 - df_new[(df_new['coin']==coin) & (df_new['model']=='GARCH11_directional')]['acc'].values
    pers_err  = 1 - df_new[(df_new['coin']==coin) & (df_new['model']=='Naive_Persistence')]['acc'].values

    for (bl_name, bl_err) in [('ARIMA(1,0,1)', arima_err),
                               ('GARCH(1,1)', garch_err),
                               ('Naive_Persistence', pers_err)]:
        n_min = min(len(ord_err), len(bl_err))
        dm_s, dm_p = dm_test_windows(ord_err[:n_min], bl_err[:n_min])
        dm_rows.append({'coin': coin, 'ordinal_model': best_model,
                         'baseline': bl_name, 'DM_stat': dm_s,
                         'p_value': dm_p, 'significant_p05': dm_p < 0.05 if dm_p == dm_p else False})

dm_df = pd.DataFrame(dm_rows)
dm_df.to_csv('data/enhanced_dm_results.csv', index=False)
print('=== DM Test: Best Ordinal vs. Baselines ===')
print(dm_df.to_string(index=False))

In [ ]:
# ── Figure: grouped bar chart ──────────────────────────────────────────────────
import seaborn as sns
sns.set_style('whitegrid')

df_new = pd.read_csv('data/enhanced_baseline_results.csv')
wf     = pd.read_csv('data/walkforward_results.csv')
bl     = pd.read_csv('data/baseline_results.csv')

# Build long-form for plotting
wf_best = (wf.groupby(['coin','model'])['acc'].mean()
            .reset_index()
            .sort_values('acc', ascending=False)
            .groupby('coin').first()
            .reset_index())
wf_best['label'] = 'Best Ordinal'

arima_m = bl.groupby('coin')['acc'].mean().reset_index()
arima_m['label'] = 'ARIMA(1,0,1)'

new_m = df_new.groupby(['coin','model'])['acc'].mean().reset_index()
label_map = {'GARCH11_directional': 'GARCH(1,1)',
             'Naive_Persistence':   'Persistence',
             'Naive_Majority':      'Majority Vote',
             'Random_Walk':         'Random Walk'}
new_m['label'] = new_m['model'].map(label_map)

plot_df = pd.concat([
    wf_best[['coin','acc','label']],
    arima_m[['coin','acc','label']],
    new_m[['coin','acc','label']]
])

ORDER  = ['Best Ordinal','ARIMA(1,0,1)','GARCH(1,1)','Persistence','Majority Vote','Random Walk']
COLORS = ['#1565C0','#E65100','#6A1B9A','#2E7D32','#C62828','#757575']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, coin in enumerate(COINS):
    ax  = axes[i]
    sub = (plot_df[plot_df['coin'] == coin]
           .set_index('label')['acc']
           .reindex(ORDER))
    bars = ax.bar(range(len(ORDER)), sub.values, color=COLORS, width=0.65, edgecolor='white')
    ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
    ax.set_title(coin, fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(ORDER)))
    ax.set_xticklabels(ORDER, rotation=38, ha='right', fontsize=7.5)
    lo = max(0.40, sub.dropna().min() - 0.02)
    hi = min(0.62, sub.dropna().max() + 0.04)
    ax.set_ylim(lo, hi)
    if i % 4 == 0:
        ax.set_ylabel('Mean Accuracy', fontsize=9)
    for bar, val in zip(bars, sub.values):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=6.5)

fig.suptitle('Walk-Forward Directional Accuracy: Ordinal Network vs. All Baselines\n(Mean over rolling test windows, 2021–2026)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig_enhanced_baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved → data/fig_enhanced_baseline_comparison.png')

In [ ]:
# ── Print LaTeX-ready Table 4 rows for the manuscript ──────────────────────────
df_new = pd.read_csv('data/enhanced_baseline_results.csv')
wf     = pd.read_csv('data/walkforward_results.csv')
bl     = pd.read_csv('data/baseline_results.csv')

print('=== TABLE 4 (LaTeX-ready) ===')
print('Coin & Best Ordinal & ARIMA(1,0,1) & GARCH(1,1) & Persistence & Majority & Rand.Walk \\\\')
for coin in COINS:
    ord_acc = wf[wf['coin']==coin].groupby('model')['acc'].mean().max()
    arima   = bl[bl['coin']==coin]['acc'].mean()
    garch   = df_new[(df_new['coin']==coin)&(df_new['model']=='GARCH11_directional')]['acc'].mean()
    pers    = df_new[(df_new['coin']==coin)&(df_new['model']=='Naive_Persistence')]['acc'].mean()
    maj     = df_new[(df_new['coin']==coin)&(df_new['model']=='Naive_Majority')]['acc'].mean()
    rw      = df_new[(df_new['coin']==coin)&(df_new['model']=='Random_Walk')]['acc'].mean()
    print(f'{coin} & {ord_acc:.4f} & {arima:.4f} & {garch:.4f} & {pers:.4f} & {maj:.4f} & {rw:.4f} \\\\')

print('\nAll outputs saved. 09_enhanced_baseline.ipynb complete.')